In [41]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import OrdinalEncoder
from sklearn import set_config
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsRegressor
from sklearn.datasets import make_regression
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression, RFECV, SelectFromModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [42]:
seguros = pd.read_csv('insurance.csv')

seguros.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [43]:
seguros['age'].max()

np.int64(64)

In [44]:
seguros.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [45]:
seguros.shape

(1338, 7)

In [46]:
seguros.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [47]:
seguros.duplicated().sum()

np.int64(1)

In [48]:
seguros_df = seguros.drop_duplicates()
seguros_df.duplicated().sum()

np.int64(0)

In [49]:
seguros_df.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1337 non-null   int64  
 1   sex       1337 non-null   str    
 2   bmi       1337 non-null   float64
 3   children  1337 non-null   int64  
 4   smoker    1337 non-null   str    
 5   region    1337 non-null   str    
 6   charges   1337 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 83.6 KB


In [50]:
limites = [0, 16, 32, 48, 64]

rotulos = ['c1', 'c2', 'c3', 'c4']

seguros_df['age'] = pd.cut(seguros_df['age'], bins=limites, labels=rotulos)


In [51]:
seguros_df.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1337 non-null   category
 1   sex       1337 non-null   str     
 2   bmi       1337 non-null   float64 
 3   children  1337 non-null   int64   
 4   smoker    1337 non-null   str     
 5   region    1337 non-null   str     
 6   charges   1337 non-null   float64 
dtypes: category(1), float64(2), int64(1), str(3)
memory usage: 74.5 KB


In [52]:
x = seguros_df
y = x.pop("charges")

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [53]:
NUMERICAS = ['bmi','children']
CATEGORICAS = ['sex','smoker','region','age']

numeric_pipe = make_pipeline(StandardScaler())

categoric_pipe = make_pipeline(OneHotEncoder(sparse_output=False, handle_unknown="ignore"))

preprocessor = ColumnTransformer(
    transformers=[
        ("num_pipe", numeric_pipe, NUMERICAS),
        ("cat_pipe", categoric_pipe, CATEGORICAS)]
)

In [54]:
pipeline_RandomForestRegressor = make_pipeline(preprocessor, RandomForestRegressor())


# Define os parâmetros que serão testados pelo GridSearchCV
param_grid_random_forest = {
    "randomforestregressor__n_estimators": [50, 100],
    "randomforestregressor__max_depth": [None, 10],
    "randomforestregressor__min_samples_split": [2, 5],
    "randomforestregressor__min_samples_leaf": [1, 2]
}

# Reduzir o número de dobras (folds( para a validação cruzada
cv = 5

# Cria a busca pelos melhores parâmetros (Grid Search)
search_RF = GridSearchCV(pipeline_RandomForestRegressor,
                         param_grid_random_forest,
                         cv=cv,
                         verbose=1,
                         error_score='raise')
# Fit 
search_RF.fit(x_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'randomforestregressor__max_depth': [None, 10], 'randomforestregressor__min_samples_leaf': [1, 2], 'randomforestregressor__min_samples_split': [2, 5], 'randomforestregressor__n_estimators': [50, 100]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimat

In [55]:
pipeline_ridge = make_pipeline(preprocessor, Ridge())

param_grid_ridge = {

    # Alpha controla a intensidade da penalização dos coeficientes.
    # Valores maiores = penalização mais forte.
    # Valores menores = penalização mais fraca.
    "ridge__alpha": [0.1, 1, 10],

    # Define se o modelo deve calcular o intercepto.
    # True = calcula o intercepto.
    # False = não calcula o intercepto.
    "ridge__fit_intercept": [True, False]
}


# Cria o Grid Search para testar todas as combinações dos hiperparâmetros definidos acima.
search_ridge = GridSearchCV(
    
    pipeline_ridge,  # Pipeline completo: pré-processamento + Ridge
    
    param_grid_ridge,     # Hiperparâmetros e valores que serão testados
    
    cv=5,                  # Validação cruzada em 5 partes (5-fold)
    
    verbose=1              # Exibe informações durante a execução
)


# Treina o Grid Search com os dados de treinamento.
# Testa as combinações, realiza a validação cruzada e identifica a melhor configuração da Ridge.
search_ridge.fit(x_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...e', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'ridge__alpha': [0.1, 1, ...], 'ridge__fit_intercept': [True, False]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to 

In [56]:
pipeline_KNeighborsRegressor = make_pipeline(preprocessor, KNeighborsRegressor())

# Define as combinações de hiperparâmetros que serão testadas
param_grid_knn = {
    # Número de vizinhos utilizados para realizar a previsão: de 2 a 9
    "kneighborsregressor__n_neighbors": range(2, 4),

    # Define o peso dos vizinhos: uniform = mesmo peso | distance = maior peso para os mais próximos
    "kneighborsregressor__weights": ['uniform', 'distance'],

    # Algoritmo utilizado para encontrar os vizinhos mais próximos
    "kneighborsregressor__algorithm": [
        'auto',       # escolhe automaticamente
        'ball_tree',  # utiliza Ball Tree
        'kd_tree',    # utiliza KD Tree
        'brute'       # calcula diretamente as distâncias
    ]
}


# Cria o Grid Search para testar as combinações de hiperparâmetros
search_knn = GridSearchCV(
    pipeline_KNeighborsRegressor,  # Pipeline completo: pré-processamento + KNN
    param_grid_knn,     # Hiperparâmetros e valores a serem testados
    cv=5,                # Validação cruzada em 5 partes (5-fold)
    verbose=1,           # Exibe informações sobre o processo
    error_score='raise'  # Interrompe se ocorrer algum erro
)


# Treina o Grid Search nos dados de treinamento
# Testa as combinações, realiza a validação cruzada e identifica a melhor configuração do KNN
search_knn.fit(x_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'kneighborsregressor__algorithm': ['auto', 'ball_tree', ...], 'kneighborsregressor__n_neighbors': range(2, 4), 'kneighborsregressor__weights': ['uniform', 'distance']}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using 

In [57]:
pipeline = make_pipeline(preprocessor, MinMaxScaler()).set_output(transform='pandas')

#Use the pipeline created above to fit and transform the train data 
x_train_trans_fit = pipeline.fit_transform(x_train, y_train)

#Transform the test set
x_test_trans_fit = pipeline.transform(x_test)

In [58]:
rf_reg = RandomForestRegressor()
rf_reg.fit(x_train_trans_fit, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the numb

In [59]:
ridge_reg = Ridge()
ridge_reg.fit(x_train_trans_fit, y_train)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [60]:
treeknn = KNeighborsRegressor(n_neighbors=3)
treeknn.fit(x_train_trans_fit, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",3
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"effective_metric_ effective_metric_: str or callableThe distance metric to use. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
"effective_metric_params_ effective_metric_params_: dictAdditional keyword arguments for the metric function. For most metricswill be same with `metric_params` parameter, but may also contain the`p` parameter value if the `effective_metric_` attribute is set to'minkowski'.",dict,{}


In [61]:
# Predict the new data 

predict_RF = rf_reg.predict(x_test_trans_fit)
predict_ridge = ridge_reg.predict(x_test_trans_fit)
predict_knn = treeknn.predict(x_test_trans_fit)

# R squared
RF_r2 = r2_score(y_test, predict_RF)
ridge_r2 = r2_score(y_test, predict_ridge)
knn_r2 = r2_score(y_test, predict_knn)



# DataFrame for performances
performances = pd.DataFrame({
    'random_forest': RF_r2,
    'ridge': ridge_r2,
    'knn': knn_r2
}, index=['transformed_MaxMin'])

performances

,random_forest,ridge,knn
transformed_MaxMin,0.862189,0.793681,0.77888


In [62]:
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error

In [63]:
#Root mean squared error score 
rmse_score_RF = mean_squared_log_error(y_test, predict_RF)
print("Random Forest Mean Square Log Error (MSLE):", rmse_score_RF)

# Mean Square Error (MSE)
mse_score_RF = mean_squared_error(y_test, predict_RF)
print("Random Forest Mean Square Error (MSE):", mse_score_RF)

# Mean Absolute Error (MAE)
mae_score_RF = mean_absolute_error(y_test, predict_RF)
print("Random Forest Mean Absolute Error (MAE):", mae_score_RF)

# Root mean squared log error score 
rmse_score_ridge = mean_squared_log_error(y_test, predict_ridge)
print("Ridge Mean Square Log Error (MSLE):", rmse_score_ridge)

# Mean Square Error (MSE)
mse_score_ridge = mean_squared_error(y_test, predict_ridge)
print("Ridge Mean Square Error (MSE):", mse_score_ridge)

# Mean Absolute Error (MAE)
mae_score_ridge = mean_absolute_error(y_test, predict_ridge)
print("Ridge Mean Absolute Error (MAE):", mae_score_ridge)

#Root mean squared log error score 
rmse_score_knn = mean_squared_log_error(y_test, predict_knn)
print("KNN Mean Square Log Error (MSLE):", rmse_score_knn)

# Mean Square Error (MSE)
mse_score_knn = mean_squared_error(y_test, predict_knn)
print("KNN Mean Square Error (MSE):", mse_score_knn)

# Mean Absolute Error (MAE)
mae_score_knn = mean_absolute_error(y_test, predict_knn)
print("KNN Mean Absolute Error (MAE):", mae_score_knn)

Random Forest Mean Square Log Error (MSLE): 0.25164153171732806
Random Forest Mean Square Error (MSE): 25323660.96645442
Random Forest Mean Absolute Error (MAE): 3159.306113121871
Ridge Mean Square Log Error (MSLE): 0.2589593665609347
Ridge Mean Square Error (MSE): 37912304.9545704
Ridge Mean Absolute Error (MAE): 4413.364313420176
KNN Mean Square Log Error (MSLE): 0.2270733659103081
KNN Mean Square Error (MSE): 40632116.11845583
KNN Mean Absolute Error (MAE): 3974.7862842325867


In [64]:
range_var_df = (pd.DataFrame({
                'Range': x_train_trans_fit.max() - x_train_trans_fit.min(),
                'Variance': x_train_trans_fit.var()})
                .sort_values(by='Variance'))

selector = VarianceThreshold(threshold=0.002)

x_train_var = selector.fit_transform(x_train_trans_fit)

# Apply the variance threshold to the scaled test set
x_test_var = selector.transform(x_test_trans_fit)

print("shape before:", x_train_trans_fit.shape)
print("shape after:", x_train_var.shape)

shape before: (1069, 13)
shape after: (1069, 13)


In [65]:
# Random Forest Regression
random_forest_reg = RandomForestRegressor()
random_forest_reg.fit(x_train_var, y_train)
random_forest_reg_pred = random_forest_reg.predict(x_test_var)

# Ridge
ridge_reg = Ridge()
ridge_reg.fit(x_train_var, y_train)
ridge_reg_pred = ridge_reg.predict(x_test_var)

# K-Nearest Neighbors
var_knn = KNeighborsRegressor(n_neighbors=1)
var_knn.fit(x_train_var, y_train)
var_knn_pred = var_knn.predict(x_test_var)



# Inicialize o DataFrame de performances
performances.loc["varThreshold_002", "knn"] = r2_score(y_test, var_knn_pred)
performances.loc["varThreshold_002", "random_forest"] = r2_score(y_test, random_forest_reg_pred)
performances.loc["varThreshold_002", "ridge"] = r2_score(y_test, ridge_reg_pred)

performances

,random_forest,ridge,knn
transformed_MaxMin,0.862189,0.793681,0.778880
varThreshold_002,0.865353,0.793681,0.693866


In [66]:
# Creando a pipeline e iniciando KBest, udando f-regression e k=10.
pipeline_kbest = make_pipeline(preprocessor,
                               MinMaxScaler(),
                               SelectKBest(score_func=f_regression, k=10)).set_output(transform='pandas') 

# Utilize o pipeline criado acima para ajustar e transformar os dados de treino
x_train_kbest = pipeline_kbest.fit_transform(x_train, y_train)

# Transforme o conjunto de teste
x_test_kbest = pipeline_kbest.transform(x_test)

In [67]:
# Random Forest Regression
random_forest_reg = RandomForestRegressor()
random_forest_reg.fit(x_train_kbest, y_train)
random_forest_reg_pred = random_forest_reg.predict(x_test_kbest)

# Ridge
ridge_reg = Ridge()
ridge_reg.fit(x_train_kbest, y_train)
ridge_reg_pred = ridge_reg.predict(x_test_kbest)

#Store the K-Nearest Neighbor
neigh = KNeighborsRegressor(n_neighbors=12)
neigh.fit(x_train_kbest, y_train)
knn_pred_kbest = neigh.predict(X = x_test_kbest)


# Para armazenar o R² de cada modelo após a seleção das 10 melhores variáveis pelo K-Best, 
# e exibir uma tabela performances permite comparar o desempenho dos modelos usando K-Best.
    

performances.loc["KBest_10", "knn"] = r2_score(y_test, knn_pred_kbest)
performances.loc["KBest_10", "random_forest"] = r2_score(y_test, random_forest_reg_pred)
performances.loc["KBest_10", "ridge"] = r2_score(y_test, ridge_reg_pred)

performances

,random_forest,ridge,knn
transformed_MaxMin,0.862189,0.793681,0.778880
varThreshold_002,0.865353,0.793681,0.693866
KBest_10,0.853286,0.792170,0.766216


In [68]:
# RFE - feature_importances_ é específico para modelos baseados em árvores, como RandomForestRegressor e DecisionTreeRegressor ou modelos de classificação
# Criando um pipeline com MinMaxScaler e RFECV para Random Forest Regressor
pipeline_rfe_forest = make_pipeline(preprocessor,
MinMaxScaler(),
RFECV(RandomForestRegressor()).set_output(transform='pandas'))

# Ajustar e transformar os dados de treino com o pipeline para RFE com Forest Regressor
x_train_rfe_forest = pipeline_rfe_forest.fit_transform(x_train, y_train)
x_test_rfe_forest = pipeline_rfe_forest.transform(x_test)

# Inicializar modelos para Regressão Linear, Random Forest Regression, SVR e Lasso
model_random_forest = RandomForestRegressor()

# Ajustar os modelos no conjunto de treino transformado
model_random_forest.fit(x_train_rfe_forest, y_train)

# Realizar previsões no conjunto de teste transformado para cada modelo
predictions_random_forest = model_random_forest.predict(x_test_rfe_forest)

# Calcular a pontuação R-quadrado (R²) para cada modelo
r2_random_forest = r2_score(y_test, predictions_random_forest)

# Atualizar o DataFrame de desempenhos para cada modelo
performances.loc["RFE", "random_forest"] = r2_random_forest

performances

,random_forest,ridge,knn
transformed_MaxMin,0.862189,0.793681,0.778880
varThreshold_002,0.865353,0.793681,0.693866
KBest_10,0.853286,0.792170,0.766216
RFE,0.861521,NaN,NaN


In [69]:
# Creating a pipeline with MinMaxScaler and RFECV for Ridge Regression
pipeline_rfe_ridge = make_pipeline(preprocessor,
                                   MinMaxScaler(),
                                   RFECV(Ridge()).set_output(transform='pandas'))

# Fit and transform the training data with the pipeline for RFE with Ridge Regression
x_train_rfe_ridge = pipeline_rfe_ridge.fit_transform(x_train, y_train)
x_test_rfe_ridge = pipeline_rfe_ridge.transform(x_test)

# Initialize Ridge Regression model
model_ridge_regression = Ridge()

# Fit the model on the transformed training set
model_ridge_regression.fit(x_train_rfe_ridge, y_train)

# Make predictions on the transformed test set for Ridge Regression
predictions_ridge_regression = model_ridge_regression.predict(x_test_rfe_ridge)

# Calculate the R-squared score for Ridge Regression
r2_ridge_regression = r2_score(y_test, predictions_ridge_regression)

# Update the performances DataFrame for Ridge Regression
performances.loc["RFE", "ridge"] = r2_ridge_regression
performances

,random_forest,ridge,knn
transformed_MaxMin,0.862189,0.793681,0.778880
varThreshold_002,0.865353,0.793681,0.693866
KBest_10,0.853286,0.792170,0.766216
RFE,0.861521,0.791347,NaN
